In [0]:
%run "/Users/l242514@lhr.nu.edu.pk/Online-retail-data-pipeline/notebooks/00_config/00_config"

In [0]:
%run "/Users/l242514@lhr.nu.edu.pk/Online-retail-data-pipeline/src/utilities/logger"

In [0]:
from pyspark.sql.functions import explode, element_at, split, current_timestamp, lit

init_logging_table(spark, TABLE_LOGS)

# Widgets for Parameterized Backfills & Runs
dbutils.widgets.text("source_path", "/Volumes/workspace/ebay_pipeline/raw/full_load/", "Source File/Folder Path")
dbutils.widgets.text("load_type", "FULL", "Load Type (FULL/INCREMENTAL)")
dbutils.widgets.text("batch_date", datetime.date.today().strftime("%Y-%m-%d"), "Batch Date (YYYY-MM-DD)")

source_path = dbutils.widgets.get("source_path")
load_type = dbutils.widgets.get("load_type").upper()
batch_date = dbutils.widgets.get("batch_date")

target_table = TABLE_BRONZE_FULL if load_type == "FULL" else TABLE_BRONZE_INC

def process_raw_to_bronze():
    # 3b. Read with Schema-on-Read + Rescue Column for Schema Drift Handling
    raw_df = (spark.read
              .schema(raw_response_schema)
              .option("multiLine", "true")
              .option("rescuedDataColumn", "_rescued_data")
              .json(source_path))

    # 3c. Explode and Enrich Metadata
    bronze_df = (raw_df
                 .select(explode("itemSummaries").alias("item"), "_rescued_data")
                 .select("item.*", "_rescued_data")
                 .withColumn("batch_id", lit(f"{load_type}_{batch_date}"))
                 .withColumn("ingestion_timestamp", current_timestamp())
                 .withColumn("ingestion_date", lit(batch_date))
                 .withColumn("load_type", lit(load_type))
                 .withColumn("source_api", lit("eBay Browse API v1"))
                 .withColumn("source_file", col("_metadata.file_path"))
                 .withColumn("source_category", element_at(split(col("_metadata.file_path"), "/"), -1))
                 .withColumn("load_timestamp", current_timestamp()))

    # Create table if it doesn't exist
    bronze_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(target_table)
    
    # Clean staging write via Idempotent MERGE / Overwrite on batch_id
    # Clear existing batch to prevent duplicate inserts on rerun
    spark.sql(f"DELETE FROM {target_table} WHERE batch_id = '{load_type}_{batch_date}'")
    
    bronze_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(target_table)
    
    count = bronze_df.count()
    return count, 0, 0  # inserted, updated, quarantined

# Execute wrapped in audit logging framework
run_step(spark, TABLE_LOGS, "Raw-to-Bronze", load_type, target_table, source_path, process_raw_to_bronze)